# Chapter 4: Implementing a GPT Model

Companion notebook for [Chapter 4 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/gpt-model). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## Setup

Code carried over from earlier chapters, plus anything this chapter needs to run. Run it once and move on.

In [ ]:
#@title Setup (run me first)

import math
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- from Chapter 2 ----
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class NextTokenDataset(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        self.inputs, self.targets = [], []
        for start in range(0, len(ids) - context_length, stride):
            window = ids[start : start + context_length + 1]
            self.inputs.append(torch.tensor(window[:-1]))   # all but the last
            self.targets.append(torch.tensor(window[1:]))   # all but the first

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


def create_dataloader(text, batch_size=8, max_length=256, stride=256,
                      shuffle=True, drop_last=True):
    dataset = NextTokenDataset(text, tiktoken.get_encoding("gpt2"),
                               context_length=max_length, stride=stride)
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, drop_last=drop_last)

# ---- from Chapter 3 ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.to_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def split_heads(self, t):
        b, n, _ = t.shape
        # (b, n, d_out) -> (b, n, heads, head_dim) -> (b, heads, n, head_dim)
        return t.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        b, n, _ = x.shape
        q = self.split_heads(self.to_q(x))
        k = self.split_heads(self.to_k(x))
        v = self.split_heads(self.to_v(x))

        scores = q @ k.transpose(2, 3)                    # (b, heads, n, n)
        scores = scores.masked_fill(self.mask[:n, :n], float('-inf'))
        weights = torch.softmax(scores / self.head_dim ** 0.5, dim=-1)
        weights = self.dropout(weights)

        out = weights @ v                                 # (b, heads, n, head_dim)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)  # concat heads
        return self.out_proj(out)                         # (b, n, d_out)

## The big picture

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # number of tokens the BPE tokenizer knows
    "context_length": 1024,  # longest input the model can look at
    "emb_dim": 768,          # length of every token vector
    "n_heads": 12,           # attention heads per block
    "n_layers": 12,          # how many transformer blocks to stack
    "drop_rate": 0.1,        # dropout probability
    "qkv_bias": True,        # bias terms in the query/key/value projections
}

In [ ]:
import torch
import torch.nn as nn

class SkeletonGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])
        # Placeholders, replaced with real components later in this chapter
        self.blocks = nn.Sequential(*[nn.Identity() for _ in range(cfg["n_layers"])])
        self.final_norm = nn.Identity()
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, token_ids):
        n_tokens = token_ids.shape[1]
        positions = torch.arange(n_tokens, device=token_ids.device)
        x = self.tok_emb(token_ids) + self.pos_emb(positions)
        x = self.drop(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)  # one score per vocabulary entry

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = torch.stack([
    torch.tensor(tokenizer.encode("Every journey begins with")),
    torch.tensor(tokenizer.encode("The weather today is")),
])

torch.manual_seed(0)
model = SkeletonGPT(GPT_CONFIG_124M)
logits = model(batch)
print(logits.shape)  # torch.Size([2, 4, 50257])

## Layer normalization

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))   # γ
        self.shift = nn.Parameter(torch.zeros(dim))  # β

    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mu) / torch.sqrt(var + self.eps)
        return self.scale * x_hat + self.shift

## Feed-forward network with GELU

### GELU instead of ReLU

In [ ]:
import math

class GELU(nn.Module):
    def forward(self, x):
        c = math.sqrt(2.0 / math.pi)
        return 0.5 * x * (1.0 + torch.tanh(c * (x + 0.044715 * x ** 3)))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.net = nn.Sequential(
            nn.Linear(d, 4 * d),  # expand:  768 → 3072
            GELU(),
            nn.Linear(4 * d, d),  # project: 3072 → 768
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
ff = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 4, 768)
print(ff(x).shape)  # torch.Size([2, 4, 768]), same shape in and out

## Shortcut connections

In [ ]:
class TinyDeepNet(nn.Module):
    def __init__(self, widths, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList(
            nn.Sequential(nn.Linear(w_in, w_out), GELU())
            for w_in, w_out in zip(widths[:-1], widths[1:])
        )

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            # A shortcut only makes sense when the shapes match
            x = x + out if self.use_shortcut and out.shape == x.shape else out
        return x

def show_gradients(model, x):
    loss = model(x).pow(2).mean()   # any scalar will do for this experiment
    loss.backward()
    for name, p in model.named_parameters():
        if name.endswith("weight"):
            print(f"{name:18s} {p.grad.abs().mean():.6f}")

x = torch.tensor([[1.0, 0.0, -1.0]])
for flag in (False, True):
    torch.manual_seed(123)
    print("shortcut =", flag)
    show_gradients(TinyDeepNet([3, 3, 3, 3, 3, 1], flag), x)

## The transformer block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.norm1 = LayerNorm(d)
        self.attn = MultiHeadAttention(
            d_in=d,
            d_out=d,
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.norm2 = LayerNorm(d)
        self.ff = FeedForward(cfg)
        self.drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # First half: tokens exchange information (attention) + shortcut
        x = x + self.drop(self.attn(self.norm1(x)))
        # Second half: each token is processed on its own (FFN) + shortcut
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

In [ ]:
block = TransformerBlock(GPT_CONFIG_124M)
x = torch.rand(2, 4, 768)
print(block(x).shape)  # torch.Size([2, 4, 768])

## The full GPT model

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, token_ids):
        n_tokens = token_ids.shape[1]
        positions = torch.arange(n_tokens, device=token_ids.device)
        x = self.tok_emb(token_ids) + self.pos_emb(positions)  # [batch, n_tokens, 768]
        x = self.drop(x)
        x = self.blocks(x)                                     # 12 blocks, shape unchanged
        x = self.final_norm(x)
        return self.out_head(x)                                # [batch, n_tokens, 50257]

### How big is it?

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

total = sum(p.numel() for p in model.parameters())
print(f"{total:,}")  # 163,037,184

In [ ]:
head = model.out_head.weight.numel()
print(f"{total - head:,}")  # 124,439,808

# To actually tie them (parameters() then counts the shared matrix once):
model.out_head.weight = model.tok_emb.weight

In [ ]:
size_mb = total * 4 / 1024**2
print(f"{size_mb:.1f} MB")  # 621.9 MB

## Generating text

In [ ]:
def generate_greedy(model, token_ids, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        window = token_ids[:, -context_size:]      # 1. crop
        with torch.no_grad():
            logits = model(window)                 # 2. [batch, n_tokens, vocab]
        last = logits[:, -1, :]                    # 3. next-token scores
        probs = torch.softmax(last, dim=-1)        # 4. probabilities...
        next_id = torch.argmax(probs, dim=-1, keepdim=True)  # ...and the winner
        token_ids = torch.cat([token_ids, next_id], dim=1)   # 5. append
    return token_ids

In [ ]:
start = tokenizer.encode("Hello, I am")
ids = torch.tensor(start).unsqueeze(0)  # add a batch dimension

model.eval()  # switch off dropout while generating
out = generate_greedy(model, ids, max_new_tokens=6,
                      context_size=GPT_CONFIG_124M["context_length"])
print(tokenizer.decode(out[0].tolist()))

## Quiz